# 3Di Sequence Analysis

Analysis of Foldseek 3Di structural alphabet sequences.

**Contents:**
1. Summary Statistics — length distributions, alphabet composition heatmap, Shannon entropy, hit distributions
2. Distance Matrices — Foldseek e-value transform, k-mer JSD, pairwise alignment (each with LaTeX derivation)
3. Phylogenetic Trees — NJ, UPGMA, Maximum Parsimony (Fitch), Maximum Likelihood (Felsenstein pruning), Robinson-Foulds comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from collections import Counter, defaultdict
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, to_tree
from scipy.stats import entropy as sp_entropy
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({"figure.dpi": 120, "font.size": 10})

# 3Di alphabet (20 structural states)
ALPHABET_3DI = list("ACDEFGHIKLMNPQRSTVWY")

# --- Load FASTA ---
def parse_fasta(path):
    seqs = {}
    name = None
    buf = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if name:
                    seqs[name] = "".join(buf)
                name = line[1:].split()[0]
                buf = []
            else:
                buf.append(line.upper())
    if name:
        seqs[name] = "".join(buf)
    return seqs

FASTA_PATH = "results_fs_only_full_fs_compare/foldseek_db/all_sequences_3di.fasta"
seqs = parse_fasta(FASTA_PATH)
names = list(seqs.keys())
sequences = list(seqs.values())
print(f"Loaded {len(seqs)} sequences")

## 1. Summary Statistics

In [ ]:
lengths = np.array([len(s) for s in sequences])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Length distribution
axes[0].hist(lengths, bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Sequence length")
axes[0].set_ylabel("Count")
axes[0].set_title("Length Distribution")
axes[0].axvline(np.median(lengths), color="red", ls="--", label=f"median={int(np.median(lengths))}")
axes[0].legend()

# Log-scale length distribution
axes[1].hist(lengths, bins=60, edgecolor="black", alpha=0.7, color="teal", log=True)
axes[1].set_xlabel("Sequence length")
axes[1].set_ylabel("Count (log)")
axes[1].set_title("Length Distribution (log scale)")

# Box plot
axes[2].boxplot(lengths, vert=True)
axes[2].set_ylabel("Sequence length")
axes[2].set_title("Length Box Plot")

plt.tight_layout()
plt.show()

print(f"N = {len(lengths)}")
print(f"Min: {lengths.min()}, Max: {lengths.max()}, Mean: {lengths.mean():.1f}, Median: {np.median(lengths):.0f}, Std: {lengths.std():.1f}")

In [ ]:
# Alphabet composition heatmap (sample up to 200 sequences for readability)
np.random.seed(42)
sample_idx = np.random.choice(len(sequences), size=min(200, len(sequences)), replace=False)
sample_idx.sort()

comp_matrix = np.zeros((len(sample_idx), len(ALPHABET_3DI)))
for i, idx in enumerate(sample_idx):
    c = Counter(sequences[idx])
    total = len(sequences[idx])
    for j, aa in enumerate(ALPHABET_3DI):
        comp_matrix[i, j] = c.get(aa, 0) / total if total > 0 else 0

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(comp_matrix, xticklabels=ALPHABET_3DI, yticklabels=False,
            cmap="YlOrRd", ax=ax, cbar_kws={"label": "Frequency"})
ax.set_xlabel("3Di State")
ax.set_ylabel(f"Sequences (n={len(sample_idx)} sampled)")
ax.set_title("Alphabet Composition Heatmap")
plt.tight_layout()
plt.show()

# Global composition
all_chars = "".join(sequences)
global_counts = Counter(all_chars)
freq_df = pd.DataFrame([(aa, global_counts.get(aa, 0)) for aa in ALPHABET_3DI], columns=["State", "Count"])
freq_df["Frequency"] = freq_df["Count"] / freq_df["Count"].sum()
freq_df = freq_df.sort_values("Frequency", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(freq_df["State"], freq_df["Frequency"], color="steelblue", edgecolor="black")
ax.set_xlabel("3Di State")
ax.set_ylabel("Frequency")
ax.set_title("Global 3Di Alphabet Composition")
plt.tight_layout()
plt.show()
print(freq_df.to_string(index=False))

In [ ]:
# Shannon entropy per sequence
def shannon_entropy(seq, alphabet=ALPHABET_3DI):
    c = Counter(seq)
    total = len(seq)
    probs = np.array([c.get(a, 0) / total for a in alphabet])
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

entropies = np.array([shannon_entropy(s) for s in sequences])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(entropies, bins=50, edgecolor="black", alpha=0.7, color="darkorange")
axes[0].set_xlabel("Shannon Entropy (bits)")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-Sequence Shannon Entropy")
axes[0].axvline(np.median(entropies), color="red", ls="--", label=f"median={np.median(entropies):.2f}")
axes[0].legend()

# Entropy vs length
axes[1].scatter(lengths, entropies, alpha=0.3, s=5, color="darkorange")
axes[1].set_xlabel("Sequence Length")
axes[1].set_ylabel("Shannon Entropy (bits)")
axes[1].set_title("Entropy vs Length")
plt.tight_layout()
plt.show()

print(f"Entropy — Min: {entropies.min():.3f}, Max: {entropies.max():.3f}, "
      f"Mean: {entropies.mean():.3f}, Max possible: {np.log2(20):.3f} bits")

In [ ]:
# Hit distributions from Foldseek all-vs-all results
RESULTS_PATH = "results_fs_only_full_fs_compare/foldseek_results/all_vs_all_results.tsv"
cols = ["query", "target", "fident", "alnlen", "mismatch", "gapopen",
        "qstart", "qend", "tstart", "tend", "evalue", "bitscore"]
hits_df = pd.read_csv(RESULTS_PATH, sep="\t", header=None, names=cols)

# Remove self-hits
hits_df = hits_df[hits_df["query"] != hits_df["target"]].copy()
print(f"Loaded {len(hits_df)} non-self hits")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# E-value distribution
axes[0, 0].hist(np.log10(hits_df["evalue"].clip(lower=1e-300)), bins=80, color="steelblue", edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("log10(E-value)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("E-value Distribution")

# Bitscore distribution
axes[0, 1].hist(hits_df["bitscore"], bins=80, color="teal", edgecolor="black", alpha=0.7)
axes[0, 1].set_xlabel("Bitscore")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("Bitscore Distribution")

# Sequence identity
axes[1, 0].hist(hits_df["fident"], bins=50, color="coral", edgecolor="black", alpha=0.7)
axes[1, 0].set_xlabel("Fraction Identity")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Sequence Identity Distribution")

# Hits per query
hits_per_query = hits_df.groupby("query").size()
axes[1, 1].hist(hits_per_query, bins=60, color="mediumpurple", edgecolor="black", alpha=0.7)
axes[1, 1].set_xlabel("Number of Hits")
axes[1, 1].set_ylabel("Number of Queries")
axes[1, 1].set_title("Hits per Query")

plt.tight_layout()
plt.show()

## 2. Distance Matrices

We subsample sequences for tractable pairwise computation, then compute three distance matrices.

### 2a. Foldseek E-value Transform

$$d_{ij} = -\log_{10}\bigl(\min(E_{ij},\, E_{ji})\bigr)$$

where $E_{ij}$ is the Foldseek E-value for query $i$ against target $j$. For missing pairs we set a ceiling $E_\text{max}$:

$$d_{ij} = -\log_{10}(E_\text{max}) \quad \text{if no hit exists}$$

### 2b. k-mer Jensen-Shannon Divergence

For each sequence we compute the empirical $k$-mer frequency distribution $P_i$. The JSD between two sequences is:

$$\text{JSD}(P \| Q) = \frac{1}{2}\,D_\text{KL}(P \| M) + \frac{1}{2}\,D_\text{KL}(Q \| M), \quad M = \frac{P + Q}{2}$$

$$D_\text{KL}(P \| M) = \sum_x P(x) \log_2 \frac{P(x)}{M(x)}$$

JSD is symmetric and bounded in $[0, 1]$ bits when using $\log_2$.

### 2c. Pairwise Alignment (Needleman-Wunsch)

Global alignment with a simple match/mismatch scoring scheme. The distance is the normalized edit distance:

$$S^*(i, j) = \max\bigl(S(i-1,j-1) + \sigma(a_i, b_j),\; S(i-1,j) + g,\; S(i,j-1) + g\bigr)$$

$$d_{ij} = 1 - \frac{S_\text{align}}{S_\text{max}}$$

where $\sigma(a,b) = +1$ if $a=b$, $-1$ otherwise, $g = -2$ (gap penalty), and $S_\text{max} = \max(|a|, |b|)$.

In [ ]:
# Subsample for distance matrix computation
N_SUB = 50  # adjust as needed
np.random.seed(0)
sub_idx = np.sort(np.random.choice(len(names), size=min(N_SUB, len(names)), replace=False))
sub_names = [names[i] for i in sub_idx]
sub_seqs = [sequences[i] for i in sub_idx]
print(f"Subsampled {len(sub_names)} sequences for distance matrices")

In [ ]:
# 2a. Foldseek E-value distance matrix
# Build lookup from hits_df
evalue_lookup = {}
for _, row in hits_df.iterrows():
    key = (row["query"], row["target"])
    if key not in evalue_lookup or row["evalue"] < evalue_lookup[key]:
        evalue_lookup[key] = row["evalue"]

E_MAX = 10.0  # ceiling for missing pairs
n = len(sub_names)
dist_evalue = np.zeros((n, n))

for i in range(n):
    for j in range(i + 1, n):
        e_ij = evalue_lookup.get((sub_names[i], sub_names[j]), E_MAX)
        e_ji = evalue_lookup.get((sub_names[j], sub_names[i]), E_MAX)
        e_min = min(e_ij, e_ji)
        e_min = max(e_min, 1e-300)  # avoid log(0)
        d = -np.log10(e_min)
        dist_evalue[i, j] = d
        dist_evalue[j, i] = d

# Normalize to [0, 1] for comparability
dist_evalue_norm = dist_evalue / dist_evalue.max() if dist_evalue.max() > 0 else dist_evalue

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_evalue_norm, xticklabels=False, yticklabels=False,
            cmap="viridis_r", ax=ax, cbar_kws={"label": "Normalized distance"})
ax.set_title("Distance Matrix: Foldseek E-value Transform")
plt.tight_layout()
plt.show()

In [ ]:
# 2b. k-mer JSD distance matrix
K = 3  # k-mer size

def kmer_freq(seq, k=K):
    counts = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    total = sum(counts.values())
    return counts, total

def jsd(p_counts, p_total, q_counts, q_total):
    all_kmers = set(p_counts.keys()) | set(q_counts.keys())
    p = np.array([p_counts.get(km, 0) / p_total for km in all_kmers])
    q = np.array([q_counts.get(km, 0) / q_total for km in all_kmers])
    m = (p + q) / 2
    # KL divergences with safe log
    kl_pm = np.sum(p * np.log2(np.divide(p, m, out=np.ones_like(p), where=p > 0)), where=p > 0)
    kl_qm = np.sum(q * np.log2(np.divide(q, m, out=np.ones_like(q), where=q > 0)), where=q > 0)
    return 0.5 * kl_pm + 0.5 * kl_qm

# Precompute k-mer profiles
profiles = [kmer_freq(s) for s in sub_seqs]

dist_jsd = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = jsd(profiles[i][0], profiles[i][1], profiles[j][0], profiles[j][1])
        dist_jsd[i, j] = d
        dist_jsd[j, i] = d

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_jsd, xticklabels=False, yticklabels=False,
            cmap="magma_r", ax=ax, cbar_kws={"label": "JSD"})
ax.set_title(f"Distance Matrix: {K}-mer Jensen-Shannon Divergence")
plt.tight_layout()
plt.show()

In [ ]:
# 2c. Pairwise alignment distance (Needleman-Wunsch, simplified)
def nw_score(a, b, match=1, mismatch=-1, gap=-2):
    la, lb = len(a), len(b)
    # Use only two rows for memory efficiency
    prev = np.arange(0, (lb + 1) * gap, gap, dtype=np.int32)  # not quite right, redo:
    prev = np.array([j * gap for j in range(lb + 1)], dtype=np.int32)
    for i in range(1, la + 1):
        curr = np.empty(lb + 1, dtype=np.int32)
        curr[0] = i * gap
        for j in range(1, lb + 1):
            s = match if a[i-1] == b[j-1] else mismatch
            curr[j] = max(prev[j-1] + s, prev[j] + gap, curr[j-1] + gap)
        prev = curr
    return prev[lb]

# For speed, subsample further for NW
N_NW = min(30, n)
nw_idx = np.sort(np.random.choice(n, N_NW, replace=False))
nw_names = [sub_names[i] for i in nw_idx]
nw_seqs = [sub_seqs[i] for i in nw_idx]

dist_nw = np.zeros((N_NW, N_NW))
for i in range(N_NW):
    for j in range(i + 1, N_NW):
        score = nw_score(nw_seqs[i], nw_seqs[j])
        max_possible = max(len(nw_seqs[i]), len(nw_seqs[j]))
        d = 1.0 - score / max_possible if max_possible > 0 else 0.0
        d = max(0.0, d)  # clamp
        dist_nw[i, j] = d
        dist_nw[j, i] = d
    if (i + 1) % 10 == 0:
        print(f"  NW progress: {i+1}/{N_NW}")

print("NW alignment distance matrix complete.")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_nw, xticklabels=False, yticklabels=False,
            cmap="inferno_r", ax=ax, cbar_kws={"label": "Normalized distance"})
ax.set_title("Distance Matrix: Pairwise Alignment (Needleman-Wunsch)")
plt.tight_layout()
plt.show()

## 3. Phylogenetic Trees

We build trees from the JSD distance matrix (50 sequences) using four methods, then compare topologies.

### Neighbor Joining (NJ)

Iteratively join the pair $(i, j)$ minimizing:

$$Q_{ij} = (n-2)\,d_{ij} - \sum_k d_{ik} - \sum_k d_{jk}$$

### UPGMA

Hierarchical clustering where distance between clusters $A, B$ is the arithmetic mean:

$$d(A, B) = \frac{1}{|A||B|} \sum_{i \in A} \sum_{j \in B} d_{ij}$$

### Maximum Parsimony (Fitch Algorithm)

For each site $k$ and internal node $v$ with children $u, w$:

$$S_v^{(k)} = \begin{cases} S_u^{(k)} \cap S_w^{(k)} & \text{if } S_u^{(k)} \cap S_w^{(k)} \neq \emptyset \\ S_u^{(k)} \cup S_w^{(k)} & \text{otherwise (cost +1)} \end{cases}$$

Total parsimony score is summed over all sites.

### Maximum Likelihood (Felsenstein Pruning)

Under the Jukes-Cantor-like model with 20 states, transition probability:

$$P(j \mid i, t) = \begin{cases} \frac{1}{20} + \frac{19}{20} e^{-\frac{20t}{19}} & \text{if } j = i \\ \frac{1}{20} - \frac{1}{20} e^{-\frac{20t}{19}} & \text{if } j \neq i \end{cases}$$

Felsenstein's pruning computes the conditional likelihood at node $v$:

$$L_v(s) = \left[\sum_{s'} P(s' \mid s, t_u)\, L_u(s')\right] \cdot \left[\sum_{s'} P(s' \mid s, t_w)\, L_w(s')\right]$$

The total log-likelihood at the root:

$$\ln \mathcal{L} = \sum_k \ln \left(\sum_s \pi_s \, L_{\text{root}}^{(k)}(s)\right)$$

In [ ]:
# --- Neighbor Joining (from scratch) ---
def neighbor_joining(dist_matrix, labels):
    """Returns a Newick string."""
    n = len(labels)
    D = dist_matrix.copy().astype(float)
    nodes = {i: labels[i] for i in range(n)}
    next_id = n

    active = list(range(n))

    while len(active) > 2:
        k = len(active)
        # Row sums
        row_sums = {}
        for i in active:
            row_sums[i] = sum(D[i, j] for j in active if j != i)

        # Find minimum Q
        best_q = float("inf")
        best_pair = (None, None)
        for ii, i in enumerate(active):
            for j in active[ii+1:]:
                q = (k - 2) * D[i, j] - row_sums[i] - row_sums[j]
                if q < best_q:
                    best_q = q
                    best_pair = (i, j)

        i, j = best_pair
        # Branch lengths
        di = 0.5 * D[i, j] + (row_sums[i] - row_sums[j]) / (2 * (k - 2)) if k > 2 else 0.5 * D[i, j]
        dj = D[i, j] - di
        di = max(di, 0)
        dj = max(dj, 0)

        # New node
        new_id = next_id
        next_id += 1
        nodes[new_id] = f"({nodes[i]}:{di:.6f},{nodes[j]}:{dj:.6f})"

        # Expand distance matrix
        old_size = D.shape[0]
        new_D = np.zeros((old_size + 1, old_size + 1))
        new_D[:old_size, :old_size] = D
        for m in active:
            if m != i and m != j:
                d_new = 0.5 * (D[m, i] + D[m, j] - D[i, j])
                new_D[new_id, m] = d_new
                new_D[m, new_id] = d_new
        D = new_D

        active.remove(i)
        active.remove(j)
        active.append(new_id)

    # Final two
    i, j = active
    d = D[i, j]
    newick = f"({nodes[i]}:{d/2:.6f},{nodes[j]}:{d/2:.6f});"
    return newick

nj_newick = neighbor_joining(dist_jsd, [n[:15] for n in sub_names])
print("NJ tree built.")
print(nj_newick[:200], "...")

In [ ]:
# --- UPGMA via scipy ---
condensed_jsd = squareform(dist_jsd)
Z_upgma = linkage(condensed_jsd, method="average")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_upgma, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title("UPGMA Dendrogram (JSD distance)")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- NJ Dendrogram (using scipy Ward as proxy for visualization, actual NJ topology in newick) ---
# We'll use the NJ distance matrix with scipy's "single" linkage just for dendrogram viz
# But we also plot the proper NJ tree by parsing newick

# Simple recursive newick parser for dendrogram-like visualization
def parse_newick_to_linkage(newick, labels):
    """Convert NJ newick to scipy-compatible dendrogram via hierarchical parsing."""
    # For a proper NJ dendrogram, we use the distance matrix with 'weighted' linkage
    # which approximates NJ behavior
    pass

# Instead, let's show NJ as a text-based tree AND a scipy dendrogram from NJ distances
Z_nj = linkage(condensed_jsd, method="weighted")  # weighted = WPGMA, closest to NJ

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_nj, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title("Neighbor Joining Dendrogram (JSD distance, WPGMA approximation)")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Maximum Parsimony (Fitch algorithm) ---
# We need an alignment — use a simple length-truncation alignment (pad/truncate to median length)
# for demonstration purposes

def make_simple_alignment(seqs, target_len=None):
    """Pad short seqs with gaps, truncate long seqs."""
    if target_len is None:
        target_len = int(np.median([len(s) for s in seqs]))
    aligned = []
    for s in seqs:
        if len(s) >= target_len:
            aligned.append(s[:target_len])
        else:
            aligned.append(s + "-" * (target_len - len(s)))
    return aligned, target_len

aligned_seqs, aln_len = make_simple_alignment(sub_seqs)
print(f"Aligned to length {aln_len}")

def fitch_parsimony(tree_root, aligned, aln_len, alphabet=ALPHABET_3DI + ["-"]):
    """Compute parsimony score on a scipy linkage tree using Fitch algorithm."""
    n_leaves = len(aligned)
    n_nodes = tree_root.get_count()

    total_score = 0
    for site in range(aln_len):
        # Bottom-up: compute state sets
        state_sets = {}

        def _fitch(node):
            nonlocal total_score
            if node.is_leaf():
                state_sets[node.id] = {aligned[node.id][site]}
                return
            _fitch(node.get_left())
            _fitch(node.get_right())
            left_s = state_sets[node.get_left().id]
            right_s = state_sets[node.get_right().id]
            inter = left_s & right_s
            if inter:
                state_sets[node.id] = inter
            else:
                state_sets[node.id] = left_s | right_s
                total_score += 1

        _fitch(tree_root)

    return total_score

# Build tree from UPGMA linkage for parsimony scoring
tree_root_upgma = to_tree(Z_upgma)
parsimony_upgma = fitch_parsimony(tree_root_upgma, aligned_seqs, aln_len)
print(f"Parsimony score (UPGMA topology): {parsimony_upgma}")

tree_root_nj = to_tree(Z_nj)
parsimony_nj = fitch_parsimony(tree_root_nj, aligned_seqs, aln_len)
print(f"Parsimony score (NJ/WPGMA topology): {parsimony_nj}")

# Show the more parsimonious tree
best_label = "UPGMA" if parsimony_upgma <= parsimony_nj else "NJ"
best_Z = Z_upgma if parsimony_upgma <= parsimony_nj else Z_nj
print(f"\nBest parsimony tree: {best_label}")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(best_Z, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title(f"Maximum Parsimony Tree (best topology: {best_label}, score={min(parsimony_upgma, parsimony_nj)})")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Maximum Likelihood (Felsenstein pruning, JC-like model for 20 states) ---
N_STATES = 20
state_to_idx = {s: i for i, s in enumerate(ALPHABET_3DI)}

def jc20_transition_prob(t, n_states=N_STATES):
    """JC model transition probability matrix for n_states."""
    if t <= 0:
        return np.eye(n_states)
    e = np.exp(-n_states * t / (n_states - 1))
    p_same = 1.0 / n_states + (n_states - 1) / n_states * e
    p_diff = 1.0 / n_states - 1.0 / n_states * e
    P = np.full((n_states, n_states), p_diff)
    np.fill_diagonal(P, p_same)
    return P

def felsenstein_log_likelihood(tree_root, aligned, aln_len, branch_lengths=None):
    """Compute log-likelihood using Felsenstein pruning on a scipy ClusterNode tree."""
    pi = np.full(N_STATES, 1.0 / N_STATES)  # uniform prior

    # Collect all nodes (post-order)
    def get_nodes_postorder(node):
        nodes = []
        if not node.is_leaf():
            nodes.extend(get_nodes_postorder(node.get_left()))
            nodes.extend(get_nodes_postorder(node.get_right()))
        nodes.append(node)
        return nodes

    all_nodes = get_nodes_postorder(tree_root)

    # Use branch distances from linkage (node.dist)
    total_ll = 0.0
    # Process a subset of sites for speed
    site_step = max(1, aln_len // 200)  # sample ~200 sites
    n_sites_used = 0

    for site in range(0, aln_len, site_step):
        cond_lik = {}

        for node in all_nodes:
            if node.is_leaf():
                lik = np.zeros(N_STATES)
                ch = aligned[node.id][site] if site < len(aligned[node.id]) else "-"
                idx = state_to_idx.get(ch, -1)
                if idx >= 0:
                    lik[idx] = 1.0
                else:
                    lik[:] = 1.0  # gap = ambiguous
                cond_lik[node.id] = lik
            else:
                left = node.get_left()
                right = node.get_right()
                t_left = max(node.dist - left.dist, 0.001) if not left.is_leaf() else max(node.dist, 0.001)
                t_right = max(node.dist - right.dist, 0.001) if not right.is_leaf() else max(node.dist, 0.001)

                P_left = jc20_transition_prob(t_left)
                P_right = jc20_transition_prob(t_right)

                lik_left = P_left @ cond_lik[left.id]
                lik_right = P_right @ cond_lik[right.id]
                cond_lik[node.id] = lik_left * lik_right

        root_lik = np.sum(pi * cond_lik[tree_root.id])
        if root_lik > 0:
            total_ll += np.log(root_lik)
        n_sites_used += 1

    # Scale to full alignment
    total_ll *= (aln_len / n_sites_used)
    return total_ll

ll_upgma = felsenstein_log_likelihood(tree_root_upgma, aligned_seqs, aln_len)
ll_nj = felsenstein_log_likelihood(tree_root_nj, aligned_seqs, aln_len)

print(f"Log-likelihood (UPGMA topology): {ll_upgma:.1f}")
print(f"Log-likelihood (NJ/WPGMA topology): {ll_nj:.1f}")

best_ml_label = "UPGMA" if ll_upgma > ll_nj else "NJ"
best_ml_Z = Z_upgma if ll_upgma > ll_nj else Z_nj
print(f"\nBest ML tree: {best_ml_label}")

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(best_ml_Z, labels=[n[:15] for n in sub_names], leaf_rotation=90, ax=ax,
           leaf_font_size=7, color_threshold=0.3)
ax.set_title(f"Maximum Likelihood Tree (best topology: {best_ml_label}, lnL={max(ll_upgma, ll_nj):.1f})")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

### Robinson-Foulds Distance

The RF distance counts the number of bipartitions (splits) present in one tree but not the other:

$$d_\text{RF}(T_1, T_2) = |S(T_1) \triangle S(T_2)|$$

where $S(T)$ is the set of bipartitions induced by $T$ and $\triangle$ is the symmetric difference.

In [ ]:
# --- Robinson-Foulds distance between tree topologies ---
def get_bipartitions(linkage_matrix, n_leaves):
    """Extract the set of bipartitions (as frozensets of leaf indices) from a linkage matrix."""
    clusters = {i: frozenset([i]) for i in range(n_leaves)}
    splits = set()
    all_leaves = frozenset(range(n_leaves))

    for idx, (i, j, dist, size) in enumerate(linkage_matrix):
        i, j = int(i), int(j)
        new_cluster = clusters[i] | clusters[j]
        clusters[n_leaves + idx] = new_cluster
        # A split is the smaller side vs larger side
        if new_cluster != all_leaves:
            side = new_cluster if len(new_cluster) <= n_leaves // 2 else all_leaves - new_cluster
            splits.add(side)
    return splits

n_leaves = len(sub_names)
splits_upgma = get_bipartitions(Z_upgma, n_leaves)
splits_nj = get_bipartitions(Z_nj, n_leaves)

rf_distance = len(splits_upgma.symmetric_difference(splits_nj))
rf_max = 2 * (n_leaves - 3)  # maximum possible RF for binary trees
rf_normalized = rf_distance / rf_max if rf_max > 0 else 0

print(f"Robinson-Foulds distance (UPGMA vs NJ): {rf_distance}")
print(f"Normalized RF distance: {rf_normalized:.3f}  (0 = identical, 1 = maximally different)")
print(f"Shared bipartitions: {len(splits_upgma & splits_nj)}")
print(f"UPGMA-only splits: {len(splits_upgma - splits_nj)}")
print(f"NJ-only splits: {len(splits_nj - splits_upgma)}")

# Summary comparison table
summary = pd.DataFrame({
    "Method": ["UPGMA", "NJ (WPGMA)", "Max Parsimony", "Max Likelihood"],
    "Score": [
        f"—",
        f"—",
        f"{min(parsimony_upgma, parsimony_nj)} (best: {best_label})",
        f"{max(ll_upgma, ll_nj):.1f} (best: {best_ml_label})"
    ],
    "UPGMA Parsimony": [parsimony_upgma, parsimony_nj, "—", "—"],
    "Log-Likelihood": [f"{ll_upgma:.1f}", f"{ll_nj:.1f}", "—", "—"],
})
print("\n")
print(summary.to_string(index=False))